# 0.04 Test site: Sentinel-1 download and preprocessing

**Purpose.** Test how achievable, how slow and how large it is to obtain Sentinel-1 input data for one
Geo-Wiki site (a 300 x 300 m area in Africa, i.e. the 100 x 100 m Geo-Wiki site plus a 100 m margin), so the cost for all sites can be extrapolated. This is the first of
several per-modality notebooks (later: Sentinel-2, SRTM, ERA5, TerraClimate, VIIRS, LandScan/location).

**Inputs.** Geo-Wiki site centre from `data/interim/geowiki/14871660/site_summary.csv` (notebook 0.03);
Google Earth Engine collection `COPERNICUS/S1_GRD`; project-wide settings in `tomer_peled_mgi_thesis.config`.

**Outputs.** Raw GeoTIFF in `data/raw/sites/{site_id}/sentinel1/`, normalised Galileo-style array in `data/processed/sites/{site_id}/`,
timing/storage tables in `runs/tables/test_site/`, log in `runs/`.

**Method source.** The Earth Engine and preprocessing code is adapted from the Galileo repository
([nasaharvest/galileo](https://github.com/nasaharvest/galileo), MIT licence, commit `0f0b5b9`):
`src/data/earthengine/s1.py` (collection, orbit/IW filtering, closest-date composite),
`src/data/earthengine/utils.py` (`get_closest_dates`), `src/data/earthengine/ee_bbox.py` (`EEBoundingBox`),
`src/data/earthengine/eo.py` (`create_ee_image` time loop, `make_combine_bands_function`, `url`-mode download) and
`src/data/dataset.py` (`_tif_to_array`, `_fillna`, `Normalizer`, `save_h5py`). Deviations are marked `# changed:`.
The temporal design follows the thesis proposal: 30-day steps from July 2015 to July 2016, no speckle filtering
and no terrain flattening.

```mermaid
flowchart LR
    A["site_summary.csv: site centre"] --> B["300 x 300 m bounding box (site + margin)"]
    B --> C["GEE COPERNICUS/S1_GRD: IW, one pass, VV/VH"]
    C --> D["12 x 30-day closest-date median composites"]
    D --> E["getDownloadURL: GeoTIFF at 10 m"]
    E --> F["data/raw/sites/{site_id}/sentinel1/ .tif"]
    F --> G["fill NaN, reshape h w t c, normalise, float16"]
    G --> H["data/processed/sites/{site_id}/ .h5"]
    E -.timed.-> I["runs/tables/test_site/ timings and storage"]
    G -.timed.-> I
```

## 1. Settings

Project-wide values come from the package config (site id, dates, step length, site size, Earth Engine login
variables). Only notebook-specific choices are set here.

| Parameter | Default | Unit | Meaning |
| --- | --- | --- | --- |
| `TEST_SITE_ID` | 1151194 | Geo-Wiki sample id | Site under test (config) |
| `DATA_START_DATE`, `DATA_END_DATE` | 2015-07-01, 2016-07-01 | date | Study window (config) |
| `DAYS_PER_TIMESTEP` | 30 | days | Composite interval (config) |
| `SITE_SIZE_METRES` | 300 | m | Site edge length (config) |
| `N_DOWNLOAD_REPEATS` | 3 | runs | Repeated download requests, to see timing variability |
| `S1_SHIFT_VALUES / S1_DIV_VALUES` | 25 / 25 | dB | Galileo's fixed Sentinel-1 normalisation |

In [ ]:
import time, shutil, warnings
from datetime import timedelta
from math import cos, radians

from googleapiclient.errors import HttpError
import ee, h5py, numpy as np, pandas as pd, rasterio, requests, xarray as xr
import rioxarray  # registers the rasterio engine
from einops import rearrange, repeat
from IPython.display import display

from tomer_peled_mgi_thesis import config as cfg
from tomer_peled_mgi_thesis.run_logging import log_run

SITE_DIR_RAW = cfg.RAW_DATA_DIR / "sites" / str(cfg.TEST_SITE_ID) / "sentinel1"
SITE_DIR_PROCESSED = cfg.PROCESSED_DATA_DIR / "sites" / str(cfg.TEST_SITE_ID)
TABLES_DIR = cfg.TABLES_DIR / "test_site"
N_DOWNLOAD_REPEATS = 3
S1_COLLECTION = "COPERNICUS/S1_GRD"
S1_BANDS = ["VV", "VH"]
S1_SHIFT_VALUES = np.array([25.0, 25.0])  # Galileo: EE S1 values roughly span -50 to 1 dB
S1_DIV_VALUES = np.array([25.0, 25.0])

for d in (SITE_DIR_RAW, SITE_DIR_PROCESSED, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
log_ctx = log_run(cfg.RUNS_DIR / f"{stamp}-test-site-sentinel1.log")
logger = log_ctx.__enter__()  # closed in the summary cell
timings = []  # rows: step, phase, seconds

class timed:
    def __init__(self, step, phase): self.step, self.phase = step, phase
    def __enter__(self): self.t = time.perf_counter()
    def __exit__(self, *exc):
        secs = time.perf_counter() - self.t
        timings.append({"step": self.step, "phase": self.phase, "seconds": secs})
        logger.info("%s [%s]: %.2f s", self.step, self.phase, secs)

logger.info("Site=%s window=%s..%s step=%dd", cfg.TEST_SITE_ID, cfg.DATA_START_DATE, cfg.DATA_END_DATE, cfg.DAYS_PER_TIMESTEP)

## 2. Earth Engine login

Uses the `EE_PROJECT` / service-account variables from the config. If no service-account key file is configured, the
persistent credentials from `earthengine authenticate` are used (as in Galileo's `get_ee_credentials`). The
`earthengine-highvolume` endpoint is used because Galileo's `url` mode does so for many small requests.
If the stored login is missing or invalid, `ee.Authenticate()` opens a one-time browser login (not timed).
If the stored login is missing or invalid, `ee.Authenticate()` opens a one-time browser login (not timed).
No credential values are printed.

In [ ]:
# changed: credentials come from project config instead of Galileo's GCP_SA_KEY / hard-coded EE_PROJECT
credentials = "persistent"  # saved user login (None would mean no credentials)
if cfg.EE_SERVICE_ACCOUNT_KEY_FILE:
    credentials = ee.ServiceAccountCredentials(cfg.EE_SERVICE_ACCOUNT_EMAIL, cfg.EE_SERVICE_ACCOUNT_KEY_FILE)

def ee_initialise():
    ee.Initialize(credentials=credentials, project=cfg.EE_PROJECT,
                  opt_url="https://earthengine-highvolume.googleapis.com")
    ee.Number(1).getInfo()  # fails here, not later, if the login is invalid

try:
    ee_initialise()
except (ee.EEException, HttpError):  # a stale token surfaces as a 401 HttpError
    if credentials != "persistent":
        raise
    ee.Authenticate(force=True)  # one-time browser login (force: replace the stale saved token); token is stored outside the project
with timed("Earth Engine initialise", "download"):
    ee_initialise()  # timed separately from any one-off login
print("Earth Engine initialised; login mode:", "persistent user credentials" if credentials == "persistent" else "service account")

## 3. Site geometry

The site centre is the Geo-Wiki site's mean location from notebook 0.03's cached summary. As in Galileo's
`EEBoundingBox.from_centre`, a box of `surrounding_metres` around the centre is built from metres-per-degree.
Galileo used 500 m (1 km tiles); here it is 150 m (300 m area: 100 m Geo-Wiki site plus margin).

In [ ]:
site = pd.read_csv(cfg.INTERIM_DATA_DIR / "geowiki" / "14871660" / "site_summary.csv",
                   usecols=["sampleid", "longitude", "latitude"]).set_index("sampleid").loc[cfg.TEST_SITE_ID]
lat, lon = float(site["latitude"]), float(site["longitude"])

def metre_per_degree(lat):  # Galileo ee_bbox.py
    m_lat = 111132.954 - 559.822 * cos(radians(2 * lat)) + 1.175 * cos(radians(4 * lat)) - 0.0023 * cos(radians(6 * lat))
    m_lon = 111412.84 * cos(radians(lat)) - 93.5 * cos(radians(3 * lat)) + 0.118 * cos(radians(5 * lat))
    return m_lat, m_lon

m_lat, m_lon = metre_per_degree(lat)
half = cfg.SITE_SIZE_METRES / 2  # changed: Galileo uses EXPORTED_HEIGHT_WIDTH_METRES / 2 = 500
min_lat, max_lat = lat - half / m_lat, lat + half / m_lat
min_lon, max_lon = lon - half / m_lon, lon + half / m_lon
polygon = ee.Geometry.Polygon([[[min_lon, min_lat], [min_lon, max_lat], [max_lon, max_lat], [max_lon, min_lat]]])
print(f"Site {cfg.TEST_SITE_ID}: lat={lat:.5f} lon={lon:.5f}")

## 4. Sentinel-1 composite graph (lazy, on Earth Engine)

Adapted from Galileo `s1.py`, `utils.get_closest_dates` and `eo.create_ee_image`: IW mode only; a single pass direction
(taken from the first image in the collection); VV and VH collections; per 30-day step, keep all images within the
interval or, if none is closer than 15 days, those nearest the interval midpoint, then take the per-band median. The
collection is queried over the window padded by 31 days so sparse steps can borrow from neighbours. The 12 composites
are stacked as bands `VV, VH, VV_1, VH_1, ...`. This only builds the request; nothing is computed until download.

In [ ]:
def date_str(d): return d if isinstance(d, str) else d.strftime("%Y-%m-%d")  # Galileo utils.date_to_string

def get_closest_dates(mid_date, imcol):  # Galileo utils.py
    fifteen_days_in_ms = 1296000000
    mid_ms = ee.Date(date_str(mid_date)).millis()
    by_dist = imcol.map(lambda im: im.set("dateDist", ee.Number(im.get("system:time_start")).subtract(mid_ms).abs()))
    by_dist = by_dist.sort("dateDist", opt_ascending=True)
    max_diff = ee.Number(by_dist.first().get("dateDist")).max(ee.Number(fifteen_days_in_ms))
    return by_dist.filterMetadata("dateDist", "not_greater_than", max_diff)

def get_s1_image_collection(region, start_date, end_date):  # Galileo s1.py
    s1 = ee.ImageCollection(S1_COLLECTION).filterDate(date_str(start_date), date_str(end_date)).filterBounds(region)
    orbit = s1.filter(ee.Filter.eq("orbitProperties_pass", s1.first().get("orbitProperties_pass"))
                      ).filter(ee.Filter.eq("instrumentMode", "IW"))
    return (orbit.filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV")),
            orbit.filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))

def get_single_s1_image(region, start_date, end_date, vv_imcol, vh_imcol):  # Galileo s1.py
    mid_date = start_date + (end_date - start_date) / 2
    kept_vv, kept_vh = get_closest_dates(mid_date, vv_imcol), get_closest_dates(mid_date, vh_imcol)
    return ee.Image.cat([kept_vv.select("VV").median(), kept_vh.select("VH").median()]).clip(region).select(S1_BANDS)

def make_combine_bands_function(bands):  # Galileo eo.py (author: Jamie Vleeshouwer)
    def combine_bands(current, previous):
        previous = ee.Image(previous)
        current = current.select(bands)
        return ee.Algorithms.If(ee.Algorithms.IsEqual(previous, None), current, previous.addBands(ee.Image(current)))
    return combine_bands

def create_s1_ee_image(region, start_date, end_date, days_per_timestep):  # Galileo eo.create_ee_image, S1 only
    vv, vh = get_s1_image_collection(region, start_date - timedelta(days=31), end_date + timedelta(days=31))
    images, cur, cur_end = [], start_date, start_date + timedelta(days=days_per_timestep)
    while cur_end <= end_date:
        images.append(get_single_s1_image(region, cur, cur_end, vv, vh))
        cur, cur_end = cur + timedelta(days=days_per_timestep), cur_end + timedelta(days=days_per_timestep)
    return ee.Image(ee.ImageCollection(images).iterate(make_combine_bands_function(S1_BANDS))), len(images)

with timed("Build Earth Engine graph", "download"):
    ee_image, n_timesteps = create_s1_ee_image(polygon, cfg.DATA_START_DATE, cfg.DATA_END_DATE, cfg.DAYS_PER_TIMESTEP)
print("Timesteps:", n_timesteps, "-> expected bands:", n_timesteps * len(S1_BANDS))

## 5. Download

As in Galileo's `EarthEngineExporter` `url` mode, `getDownloadURL` requests a GeoTIFF at 10 m scale and the response is
streamed to disk. The request is repeated `N_DOWNLOAD_REPEATS` times (the same file is overwritten) to expose latency
variability; the first request also includes server-side graph evaluation, which is the cost that matters for a
first-time site. The GEE-computed composites are the "raw" input; the only edit is adding band names (below). Files are named `<site>_sentinel1_<start>_<end>.tif`.

In [ ]:
tif_path = SITE_DIR_RAW / f"{cfg.TEST_SITE_ID}_sentinel1_{cfg.DATA_START_DATE}_{cfg.DATA_END_DATE}.tif"  # changed: Galileo names files by bbox
download_secs = []
for i in range(N_DOWNLOAD_REPEATS):
    t0 = time.perf_counter()
    url = ee_image.getDownloadURL({"region": polygon, "scale": 10, "filePerBand": False, "format": "GEO_TIFF"})
    t_url = time.perf_counter()
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with tif_path.open("wb") as f:
        shutil.copyfileobj(r.raw, f)
    t1 = time.perf_counter()
    timings += [{"step": f"Request download URL (run {i+1})", "phase": "download", "seconds": t_url - t0},
                {"step": f"Transfer GeoTIFF (run {i+1})", "phase": "download", "seconds": t1 - t_url}]
    download_secs.append(t1 - t0)
    logger.info("Download run %d: url %.2f s, transfer %.2f s, %d bytes", i + 1, t_url - t0, t1 - t_url, tif_path.stat().st_size)
print("Download seconds per run:", [round(s, 2) for s in download_secs])

# changed: Galileo's GeoTIFFs have no band names; label each band "<VV|VH>_<step start date>" (t-major, c-minor order)
step_starts = [cfg.DATA_START_DATE + timedelta(days=cfg.DAYS_PER_TIMESTEP * t) for t in range(n_timesteps)]
band_names = [f"{b}_{d}" for d in step_starts for b in S1_BANDS]
with rasterio.open(tif_path, "r+") as src:
    src.descriptions = band_names
    print(src.count, "bands:", src.descriptions[:4], "...", src.descriptions[-2:])

## 6. Preprocessing

Mirrors Galileo's `Dataset._tif_to_array`: read the GeoTIFF, reshape `(t c) h w -> h w t c`, replace NaN/Inf with the
per-band mean (`_fillna`; all-NaN bands become 0), normalise with `(x - 25) / 25` and cast to float16, then save to HDF5
(`save_h5py`). `changed:` the NDVI/S2/climate parts are omitted (other notebooks), and the fixed shift/div values are
used (Galileo's `std=False` path) since the `normalization.json` mean/std file is not available for this dataset.
The month index array is derived from the start date in the filename as in `month_array_from_file`.

In [ ]:
def fillna(data, n_bands):  # Galileo dataset.py _fillna (4D branch)
    if data.shape[-1] != n_bands:
        raise ValueError(f"Expected {n_bands} bands - got {data.shape[-1]}")
    is_bad = np.isnan(data) | np.isinf(data)
    if not is_bad.any():
        return data
    data = np.nan_to_num(data, nan=np.nan, posinf=np.nan, neginf=np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean_per_time_band = np.nanmean(data, axis=(0, 1))  # t, b
    mean_per_time_band = np.nan_to_num(mean_per_time_band, nan=0, posinf=0, neginf=0)
    fill = repeat(np.nanmean(mean_per_time_band, axis=0), "b -> h w t b",
                  h=data.shape[0], w=data.shape[1], t=data.shape[2]) * is_bad
    return np.nan_to_num(data, nan=0, posinf=0, neginf=0) + fill

with timed("Read GeoTIFF", "preprocess"):
    with xr.open_dataarray(tif_path, engine="rasterio") as da:  # Galileo uses rioxarray.open_rasterio
        values = da.values  # [bands, H, W]
with timed("Reshape, fill NaN, normalise, float16", "preprocess"):
    s1 = rearrange(values, "(t c) h w -> h w t c", c=len(S1_BANDS))
    n_nan = int(np.isnan(s1).sum())
    s1 = fillna(s1, len(S1_BANDS))
    s1 = ((s1 - S1_SHIFT_VALUES) / S1_DIV_VALUES).astype(np.half)
    start_month = cfg.DATA_START_DATE.month  # changed: file name no longer carries the date
    months = np.fmod(np.arange(start_month - 1, start_month - 1 + s1.shape[2]), 12)
h5_path = SITE_DIR_PROCESSED / "sentinel1.h5"
with timed("Save HDF5", "preprocess"):
    with h5py.File(h5_path, "w") as hf:  # Galileo save_h5py (S1 only; dataset name kept as s_t_x)
        hf.create_dataset("s_t_x", data=s1)
        hf.create_dataset("months", data=months)
print(f"Array [h, w, t, c] = {s1.shape}, dtype {s1.dtype}; NaN/Inf cells before fill: {n_nan}; months: {months.tolist()}")

## 7. Timing and storage summary

Totals split into download (login, graph, URL request, transfer) and preprocessing. Storage is on-disk file size of the
raw GeoTIFF and the processed HDF5. The extrapolation multiplies this single-site, sequential result by a number of
sites; it ignores parallel requests, GEE quota limits and site-to-site variation, so treat it as a rough scale only.

In [ ]:
t = pd.DataFrame(timings)
first_download = t[t.step.str.contains("run 1|initialise|graph")].seconds.sum()
mean_repeat = float(np.mean(download_secs))
pre = t[t.phase == "preprocess"].seconds.sum()
storage = pd.DataFrame({"file": [tif_path.name, h5_path.name], "kind": ["raw GeoTIFF", "processed HDF5"],
                        "bytes": [tif_path.stat().st_size, h5_path.stat().st_size]})
storage["MB"] = storage.bytes / 1e6
summary = pd.DataFrame([
    {"metric": "Download incl. login, graph, first request (s)", "value": first_download},
    {"metric": "Mean download per request (s)", "value": mean_repeat},
    {"metric": "Preprocessing (s)", "value": pre},
    {"metric": "Per-site time, steady state (s)", "value": mean_repeat + pre},
    {"metric": "Raw GeoTIFF (MB)", "value": storage.MB[0]},
    {"metric": "Processed HDF5 (MB)", "value": storage.MB[1]},
])
display(t.round(3)); display(storage[["kind", "bytes", "MB"]].round(4)); display(summary.round(3))
N = 1000
per_site = mean_repeat + pre
print(f"Extrapolation per {N} sites, sequential: {per_site * N / 3600:.2f} h; "
      f"raw {storage.MB[0] * N / 1e3:.2f} GB; processed {storage.MB[1] * N / 1e3:.2f} GB")
t.assign(site_id=cfg.TEST_SITE_ID, modality="sentinel1").to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_sentinel1_timings.csv", index=False)
storage.assign(site_id=cfg.TEST_SITE_ID, modality="sentinel1").to_csv(TABLES_DIR / f"{cfg.TEST_SITE_ID}_sentinel1_storage.csv", index=False)
logger.info("Per-site steady-state %.2f s; raw %d B; processed %d B", per_site, storage.bytes[0], storage.bytes[1])
log_ctx.__exit__(None, None, None);

## 8. Conclusions

Result of the run on 2026-10-28 (single site, one machine, sequential; timings vary between requests):

- **Achievable.** Earth Engine returned all 12 composites (24 bands, 31 x 31 pixels at 10 m for the 300 x 300 m area) with
  plausible backscatter (VV about -18 to -3 dB, VH about -33 to -10 dB) and no missing pixels.
- **Time.** About 4 s per site for download (URL request about 2 s, transfer about 2 s; the first run also paid about 2 s
  login and graph set-up) plus about 0.2 s preprocessing, so roughly 1.1 h per 1,000 sites if run one after another.
- **Storage.** About 188 KB raw GeoTIFF and 48 KB processed float16 HDF5 per site, i.e. about 0.19 GB and 0.05 GB per 1,000 sites.
- **Coverage.** A separate check found 32 IW images (all ascending, VV+VH) over the site between June 2015 and July 2016,
  about every 12 days, with gaps of 5 weeks (late Oct to end Nov 2015) and 3 weeks (Jan to Feb 2016). Every 30-day step
  therefore contains at least one acquisition or a nearby one.
- **Not yet assessed.** Request limits for many sites, parallel downloads, and variation between sites and regions.